# Generate a RefCov based on Custom Channels

Finds the closest channels in the predefined leadfield matrix and computes the refCov ready to use. This part of the pygedai package requires additional installs.

In [ ]:
!pip install numpy
!pip install pandas
!pip install scipy
!pip install mat73

!pip install mne # for loading a standard montage, can also be replaced with SFP file, load as DF.

In [ ]:
import torch
from pathlib import Path
import scipy
import numpy as np

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from pygedai.ref_cov import interpolate_ref_cov
from pygedai.GEDAI import gedai

# Load Electrode Positions (Channel Labels, X, Y, Z)

In [ ]:
PATH_TO_EEG = "myeegexample.mat" # version < 7.2 for scipy
REMOVE_REFERENCE_CHANNEL = True # Assumes that reference channel is at position -1
SAMPLING_RATE_KEY = "samplingRate" # key to Frequency for eeg in PATH_TO_EEG file

# assumes at least this amount of samples are present, just to find out which part of the 
# dict is actually the EEG
MIN_SAMPLES_PRESENT = 200

In [ ]:
def load_channel_positions():
    import mne
    import pandas as pd

    if REMOVE_REFERENCE_CHANNEL: # 128 by default
        montage = mne.channels.make_standard_montage("GSN-HydroCel-128")
    else:
        montage = mne.channels.make_standard_montage("GSN-HydroCel-129")
        
    montage_head = mne.channels.transform_to_head(montage)
    ch_pos = montage_head.get_positions()["ch_pos"] # dict: {name -> array([x, y, z])}
    df = (
        pd.DataFrame.from_dict(ch_pos, orient="index", columns=["X", "Y", "Z"])
          .reset_index()
          .rename(columns={"index": "channel_name"})
    )
    return df
electrode_positions = load_channel_positions()
electrode_positions.head()

In [ ]:
ref_cov = interpolate_ref_cov(electrode_positions, dtype=torch.float64)

# Load EEG

In [ ]:
egg_mat_dict = scipy.io.loadmat(str(PATH_TO_EEG))
raw_eeg = None
sfreq = float(egg_mat_dict[SAMPLING_RATE_KEY].item())
print(f"Found sample frequency: {sfreq}")

for k, v in egg_mat_dict.items():
    if not isinstance(v, np.ndarray) or len(v.shape) != 2:
        continue

    if v.shape[0] == 129 and v.shape[1] > MIN_SAMPLES_PRESENT:
        print(f"Found EEG at key: {k} with shape {v.shape}")
        raw_eeg = torch.from_numpy(v.astype(np.float64))
        if REMOVE_REFERENCE_CHANNEL:
            print("WARNING: Removing Reference (e.g. Cz) Channel at Position -1")
            raw_eeg = raw_eeg[:-1]
            print(raw_eeg.shape)
        break

# Use GEDAI

In this version only return cleaned tensor, no other scores. Doesn't use ENOVA so input lenght = output length. ENOVA threshold of e.g. 0.9 would reduce noise but removes to noisy samples meaning input length != output length.

In [ ]:
clean_eeg = gedai(
    raw_eeg, 
    sfreq, 
    denoising_strength='auto', # GEDAI default 
    leadfield=ref_cov, 
    device='cpu', 
    dtype=torch.float32, # highest percision float64, float32 faster (around 2x)
    skip_checks_and_return_cleaned_only=True # only return clean tensor, no dict, saves a lot of time
)
clean_eeg.shape

In [ ]:
clean_eeg.min(), clean_eeg.max()

In [ ]:
import matplotlib.pyplot as plt

NUM_SEC = -1 # How many seconds off EEG to show, -1 = disable

y = clean_eeg.detach().cpu().float().numpy()
x = raw_eeg.detach().cpu().float().numpy()

def minmax(a, axis=-1, eps=1e-12):
    a_min = a.min(axis=axis, keepdims=True)
    a_max = a.max(axis=axis, keepdims=True)
    return (a - a_min) / (a_max - a_min + eps)

# normalize per-channel over time
y = minmax(y, axis=1)
x = minmax(x, axis=1)

L = 5
t = np.arange(y.shape[1]) / float(sfreq)
t_max = None if NUM_SEC == -1 else int(NUM_SEC*sfreq)

fig, axes = plt.subplots(L, 1, figsize=(10, 1.8*L*3), sharex=True)

for i in range(L):
    x_i = x[i]
    y_i = y[i]
    t_i = t
    
    if t_max is not None:
        x_i = x_i[:t_max]
        y_i = y_i[:t_max]
        t_i = t[:t_max]
    axes[i].plot(t_i, x_i, color='gray', label='Raw EEG', linewidth=1)
    axes[i].plot(t_i, y_i, color='blue', label='GEDAI EEG', linewidth=1)
    axes[i].set_ylabel(f"Channel {i}")
    if i == 0:
        axes[i].legend(loc='upper right')

axes[-1].set_xlabel("Time (s)")
fig.tight_layout()
plt.show()